# 7. Working with Stellar Catalogs

In this tutorial, you will learn how to query the internal database for identified stars and inspect their cross-session observational data.


## 7.1 Querying the Stellar Registry

The `astrometrics.stars` registry stores cross-session data for stars detected across all your targets.


In [1]:
from astrometricslib import Astrometrics

astrometrics = Astrometrics()

# Get a list of all identified stellar objects
stars = astrometrics.stars.list_objects()
print(f"Total stars tracked in the database: {len(stars)}")

if stars:
    # Inspect the first star
    star = stars[0]
    print(f"Star ID: {star.id}")
    print(f"Coordinates: RA {star.right_ascension}, DEC {star.declination}")
    print(f"Number of observing sessions matched: {len(star.session_matches)}")

Total stars tracked in the database: 562
Star ID: NGC  6205   620
Coordinates: RA 250.42648819042805, DEC 36.46399056790896
Number of observing sessions matched: 0


## 7.2 Inspecting Light Curves

A `StellarObject` may contain a `PhotometryResult` object if photometry has been run on it.


In [2]:
# Find the first star that actually has light-curve data
star_with_light_curve = next(
    (candidate_star for candidate_star in stars if candidate_star.photometry and candidate_star.photometry.timestamps),
    None,
)

if star_with_light_curve:
    light_curve = star_with_light_curve.photometry
    print(f"Star ID: {star_with_light_curve.id}")
    print(f"Light curve points: {len(light_curve.timestamps)}")
    print(f"First flux value: {light_curve.fluxes[0]:.4f}")
else:
    print("No light curve data found for any star.")

Star ID: HD 150998
Light curve points: 5
First flux value: 52220.4862


## 7.3 A Star's Spectrum Over Time

A `StellarObject` may also contain a `SpectroscopyResult` object if spectroscopy has been run on it. Alongside it, `spectra_history` holds one `SpectralObservation` snapshot per observing session -- each time the spectroscopy pipeline processes a new session for the same star, a new entry is appended, while re-processing an already-seen session updates its existing entry in place rather than duplicating it. The bundled sample data only contains a single spectroscopy session, so the star below has exactly one entry, but the same list keeps growing as more sessions are collected.

In [3]:
# Find the first star that actually has spectroscopy data
star_with_spectrum = next(
    (candidate_star for candidate_star in stars if candidate_star.spectroscopy and candidate_star.spectroscopy.wavelengths_angstrom),
    None,
)

if star_with_spectrum:
    spectroscopy_result = star_with_spectrum.spectroscopy  # a SpectroscopyResult object
    print(f"Star ID: {star_with_spectrum.id}")
    print(f"Wavelength points: {len(spectroscopy_result.wavelengths_angstrom)}")
    print(f"Spectral observation sessions recorded: {len(star_with_spectrum.spectra_history)}")
    latest_observation = star_with_spectrum.spectra_history[-1]
    print(f"Latest session timestamp: {latest_observation.timestamp}")
else:
    print("No spectroscopy data found for any star.")

Star ID: BD+35  2851::spectroscopy
Wavelength points: 416
Spectral observation sessions recorded: 1
Latest session timestamp: 2026-05-24 05:33:55.358000+00:00
